# 05a - Précalculer la série temporelle EXIOBASE

Ce notebook prépare un cache compact pour l’analyse du découplage. Chaque table annuelle est téléchargée, vérifiée, traitée séparément, puis libérée de la mémoire.

Le résultat contient 49 lignes par année, une pour chaque région EXIOBASE. Il évite de conserver les grandes matrices $L$, $M$ ou $D_{cba}$ dans un fichier intermédiaire.

## 1. Pourquoi ne pas calculer l’inverse de Leontief ?

Pour un seul indicateur climatique, nous cherchons le multiplicateur $m = s_{climat}L$. Il vérifie :

$$(I-A)^T m^T = s_{climat}^T$$

Résoudre ce système donne directement $m$ sans construire toute la matrice $L=(I-A)^{-1}$. L’empreinte régionale est ensuite obtenue par $mY_r$, à laquelle sont ajoutées les émissions directes de la demande finale.

Cette formulation réduit fortement la mémoire utilisée et accélère le calcul lorsque seuls quelques indicateurs sont nécessaires.

### Contenu du cache

Pour chaque année et région, nous conservons :

- l’empreinte climatique de consommation ;
- les émissions associées à la production observée ;
- les émissions de production attribuables par le modèle ;
- les émissions directes de la demande finale ;
- le PIB calculé avec les facteurs de production d’EXIOBASE ;
- la population fournie avec la série ;
- les métadonnées permettant de reproduire le calcul.

Le PIB EXIOBASE est exprimé en prix courants. Il sera conservé comme contrôle, mais ne sera pas utilisé comme mesure de croissance réelle dans le notebook `05`.

In [ ]:
from hashlib import sha256
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import pymrio as mr
import requests

print(f"PyMRIO {mr.__version__}, NumPy {np.__version__}, pandas {pd.__version__}")

## 2. Configurer la série

EXIOBASE 3.10.2 fournit 28 tables annuelles de 1995 à 2022. Les archives `ixi` occupent ensemble environ 7 Go. Elles sont lues directement au format ZIP et ne sont jamais décompressées sur le disque.

Pour un premier test, remplacez `ANNEES_CIBLES` par `[2022]`. Le traitement complet peut ensuite reprendre sans recalculer les années déjà présentes dans le cache.

In [ ]:
VERSION_EXIOBASE = "3.10.2"
IDENTIFIANT_ZENODO = "20051562"
SYSTEME = "ixi"
ANNEES_CIBLES = list(range(1995, 2023))

DOSSIER_EXIOBASE = Path("D:/EXIOBASE/3.10.2")
DOSSIER_CACHE = DOSSIER_EXIOBASE / "serie_temporelle"
FICHIER_CACHE = DOSSIER_CACHE / "comptes_climat_regionaux.csv"
FICHIER_HASHES = DOSSIER_EXIOBASE / "hashes.csv"
FICHIER_POPULATION = DOSSIER_EXIOBASE / "exiobase_population.txt"
URL_BASE = f"https://zenodo.org/api/records/{IDENTIFIANT_ZENODO}/files"

DOSSIER_EXIOBASE.mkdir(parents=True, exist_ok=True)
DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)

## 3. Télécharger et vérifier les fichiers

Zenodo publie une somme SHA256 pour chaque archive. Le téléchargement utilise un fichier temporaire portant l’extension `.part`. Une archive interrompue ne peut donc pas être confondue avec un fichier complet.

Les archives existantes sont conservées. Ce notebook ne supprime aucune table EXIOBASE.

In [ ]:
if not FICHIER_HASHES.exists():
    reponse = requests.get(f"{URL_BASE}/hashes.csv/content", timeout=60)
    reponse.raise_for_status()
    FICHIER_HASHES.write_bytes(reponse.content)

hashes = pd.read_csv(FICHIER_HASHES).set_index("file")
noms_attendus = {f"IOT_{annee}_{SYSTEME}.zip" for annee in ANNEES_CIBLES}
assert noms_attendus <= set(hashes.index)
hashes.loc[sorted(noms_attendus)].head()

In [ ]:
def calculer_sha256(chemin, taille_bloc=4 * 1024 * 1024):
    somme = sha256()
    with chemin.open("rb") as fichier:
        for bloc in iter(lambda: fichier.read(taille_bloc), b""):
            somme.update(bloc)
    return somme.hexdigest()


def obtenir_archive(annee):
    nom = f"IOT_{annee}_{SYSTEME}.zip"
    chemin = DOSSIER_EXIOBASE / nom
    attendu = hashes.loc[nom, "hash_sha256"]

    if chemin.exists() and calculer_sha256(chemin) == attendu:
        print(f"{annee} : archive déjà présente et valide")
        return chemin, attendu

    temporaire = chemin.with_suffix(chemin.suffix + ".part")
    url = f"{URL_BASE}/{nom}/content"
    print(f"{annee} : téléchargement de {nom}")

    with requests.get(url, stream=True, timeout=(30, 120)) as reponse:
        reponse.raise_for_status()
        taille = int(reponse.headers.get("Content-Length", 0))
        recus = 0
        prochain_message = 10
        with temporaire.open("wb") as fichier:
            for bloc in reponse.iter_content(chunk_size=4 * 1024 * 1024):
                if not bloc:
                    continue
                fichier.write(bloc)
                recus += len(bloc)
                if taille:
                    pourcentage = 100 * recus / taille
                    if pourcentage >= prochain_message:
                        print(f"  {pourcentage:3.0f} %")
                        prochain_message += 10

    if calculer_sha256(temporaire) != attendu:
        raise ValueError(f"Somme SHA256 incorrecte pour {nom}")
    temporaire.replace(chemin)
    return chemin, attendu

## 4. Définir le calcul annuel ciblé

Les facteurs PRG100 sont ceux du GIEC AR6 déjà utilisés dans les notebooks précédents. Les agrégats HFC et PFC restent exclus faute de composition détaillée.

Le système linéaire est résolu avec NumPy. Le test sur 2022 prend environ une minute dans l’environnement de ce projet, dont moins de douze secondes pour la résolution elle-même.

In [ ]:
candidats_script = [
    Path("calculer_annee_decouplage.py"),
    Path("miniprojet_mines/empreinte/calculer_annee_decouplage.py"),
]
SCRIPT_CALCUL = next(
    (chemin.resolve() for chemin in candidats_script if chemin.exists()), None
)
if SCRIPT_CALCUL is None:
    raise FileNotFoundError("Script calculer_annee_decouplage.py introuvable.")
SCRIPT_CALCUL

In [ ]:
population = pd.read_csv(
    FICHIER_POPULATION, sep="\t", index_col="Year"
)
assert set(ANNEES_CIBLES) <= set(population.index)
population.loc[ANNEES_CIBLES, ["FR", "DE", "US"]].head()

In [ ]:
def calculer_annee_isolee(annee, archive, empreinte_sha256):
    sortie_annuelle = DOSSIER_CACHE / f"comptes_{annee}.csv.part"
    commande = [
        sys.executable, str(SCRIPT_CALCUL),
        "--annee", str(annee),
        "--archive", str(archive),
        "--population", str(FICHIER_POPULATION),
        "--sortie", str(sortie_annuelle),
        "--sha256", empreinte_sha256,
        "--version", VERSION_EXIOBASE,
        "--systeme", SYSTEME,
    ]
    subprocess.run(commande, check=True)
    resultat = pd.read_csv(sortie_annuelle).set_index(["annee", "region"])
    sortie_annuelle.unlink()
    return resultat

## 5. Calculer avec reprise automatique

Le cache est relu au démarrage. Une année déjà calculée avec la même version est ignorée. Après chaque nouvelle année, le tableau complet est écrit dans un fichier temporaire puis remplace le cache précédent.

Ainsi, une interruption du téléchargement ou du calcul ne fait perdre au maximum que l’année en cours.

In [ ]:
if FICHIER_CACHE.exists():
    cache = pd.read_csv(FICHIER_CACHE).set_index(["annee", "region"])
    cache = cache.loc[cache["version_exiobase"].eq(VERSION_EXIOBASE)]
else:
    cache = pd.DataFrame()

annees_deja_calculees = (
    set(cache.index.get_level_values("annee")) if not cache.empty else set()
)
print(f"Années déjà calculées : {sorted(annees_deja_calculees)}")

In [ ]:
for annee in ANNEES_CIBLES:
    if annee in annees_deja_calculees:
        print(f"{annee} : résultat déjà présent")
        continue

    archive, empreinte_sha256 = obtenir_archive(annee)
    nouveau = calculer_annee_isolee(annee, archive, empreinte_sha256)
    cache = pd.concat([cache, nouveau]).sort_index()

    temporaire = FICHIER_CACHE.with_suffix(".csv.part")
    cache.to_csv(temporaire)
    temporaire.replace(FICHIER_CACHE)
    print(
        f"{annee} : {len(nouveau)} régions calculées en "
        f"{nouveau['temps_calcul_s'].iloc[0]:.1f} s"
    )

## 6. Contrôler le cache

Chaque année doit contenir les mêmes 49 régions, sans doublon ni valeur manquante dans les grandeurs principales. La différence entre production observée et attribuable quantifie les pressions associées aux colonnes de production monétaire nulle.

In [ ]:
cache = pd.read_csv(FICHIER_CACHE).set_index(["annee", "region"]).sort_index()
cache_cible = cache.loc[cache.index.get_level_values("annee").isin(ANNEES_CIBLES)]
colonnes_principales = [
    "ghg_cba_kgco2e",
    "ghg_pba_observe_kgco2e",
    "ghg_pba_attribuable_kgco2e",
    "pib_exiobase_meur_courants",
    "population",
]

assert not cache_cible.index.duplicated().any()
assert cache_cible[colonnes_principales].notna().all().all()
assert cache_cible.groupby(level="annee").size().eq(49).all()
assert set(ANNEES_CIBLES) <= set(cache_cible.index.get_level_values("annee"))

pd.DataFrame(
    {
        "années": [cache_cible.index.get_level_values("annee").nunique()],
        "régions par année": [cache_cible.groupby(level="annee").size().iloc[0]],
        "taille du cache (Mo)": [FICHIER_CACHE.stat().st_size / 1024**2],
        "temps cumulé de calcul (min)": [
            cache_cible.groupby(level="annee")["temps_calcul_s"].first().sum() / 60
        ],
    }
).round(2)

In [ ]:
france = cache_cible.xs("FR", level="region")[[
    "ghg_cba_kgco2e",
    "ghg_pba_observe_kgco2e",
    "pib_exiobase_meur_courants",
    "population",
]].copy()
france[["ghg_cba_kgco2e", "ghg_pba_observe_kgco2e"]] *= 1e-9
france = france.rename(columns={
    "ghg_cba_kgco2e": "empreinte (Mt CO2e)",
    "ghg_pba_observe_kgco2e": "production observée (Mt CO2e)",
})
france.round(2)

## À retenir

- il n’est pas nécessaire de calculer une inverse complète pour obtenir un multiplicateur environnemental ;
- le traitement annuel limite l’usage de la mémoire ;
- un cache minimal est plus facile à contrôler qu’une sauvegarde de toutes les matrices ;
- les sommes de contrôle sécurisent les téléchargements ;
- la sauvegarde après chaque année permet de reprendre un calcul interrompu ;
- le PIB en prix courants ne permet pas, à lui seul, de mesurer correctement le découplage dans le temps.